# Taller 1: Sistema masa-resorte-amortiguador de 2 GDL con excitacion de base

Este notebook describe y reproduce el proceso del README en la carpeta Taller 1.

## Objetivo general
Modelar y simular la dinamica vertical de un sistema mecanico de 2 GDL con entrada de base senoidal y analizar la respuesta de x1(t) y x2(t).

Voy a modificar ahora mismo la celda de ecuaciones en la notebook para dejarla en formato LaTeX como indicaste, y enseguida te confirmo el cambio exacto y la ruta.

## 1. Descripcion del sistema

- m1: masa no suspendida (rueda/eje).
- m2: masa suspendida (chasis).
- k1: rigidez del neumatico respecto a la base.
- k2: rigidez de la suspension entre m1 y m2.
- c: amortiguamiento viscoso entre m1 y m2.
- u(x): perfil irregular del camino en funcion del espacio.
- y(t): entrada de base en el tiempo, definida como y(t)=u(x(t)).

## 2. Ecuaciones diferenciales

Ecuaciones del sistema:

$$
m_1 \ddot{x}_1 + c(\dot{x}_1-\dot{x}_2) + k_2(x_1-x_2) + k_1\big(x_1-u(x(t))\big)=0
$$

$$
m_2 \ddot{x}_2 + c(\dot{x}_2-\dot{x}_1) + k_2(x_2-x_1)=0
$$

## 3. Forma matricial

M*x_ddot + C*x_dot + K*x = f(t)

M = [[m1, 0], [0, m2]]
C = [[c, -c], [-c, c]]
K = [[k1+k2, -k2], [-k2, k2]]
f(t) = [k1*u(x(t)), 0]^T

## 4. Condiciones iniciales

Para simulacion desde reposo:

x1(0)=0, x1_dot(0)=0, x2(0)=0, x2_dot(0)=0

In [ ]:
# Importaciones
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

In [ ]:
# 5. Definicion de parametros (ajustables)
m1 = 40.0      # kg
m2 = 300.0     # kg
k1 = 180000.0  # N/m
k2 = 18000.0   # N/m
c = 1200.0     # N*s/m

v = 10.0       # m/s velocidad longitudinal constante

t0, tf = 0.0, 10.0
t_eval = np.linspace(t0, tf, 3000)

In [ ]:
# 6. Modelo en espacio de estado con perfil espacial irregular
# Estado: z = [x1, x1_dot, x2, x2_dot]

def x_longitudinal(t):
    return v * t

def u_camino(x):
    # Perfil irregular espacial (ejemplo sintetico)
    return (
        0.010*np.sin(0.9*x) +
        0.006*np.sin(2.8*x + 0.7) +
        0.003*np.sin(6.5*x + 1.4)
    )

def y_base(t):
    return u_camino(x_longitudinal(t))

def modelo(t, z):
    x1, x1_dot, x2, x2_dot = z
    y = y_base(t)

    x1_ddot = (-c*(x1_dot - x2_dot) - k2*(x1 - x2) - k1*(x1 - y)) / m1
    x2_ddot = (-c*(x2_dot - x1_dot) - k2*(x2 - x1)) / m2
    return [x1_dot, x1_ddot, x2_dot, x2_ddot]

In [ ]:
# 7. Simulacion
z0 = [0.0, 0.0, 0.0, 0.0]
sol = solve_ivp(modelo, (t0, tf), z0, t_eval=t_eval, method='RK45')

t = sol.t
x = x_longitudinal(t)
x1 = sol.y[0]
x2 = sol.y[2]
y = y_base(t)

In [ ]:
# 8. Graficas principales
fig, ax = plt.subplots(2, 1, figsize=(10, 7))

ax[0].plot(x, y, color='tab:gray', linewidth=1.5)
ax[0].set_title('Perfil irregular del camino u(x)')
ax[0].set_xlabel('Espacio x [m]')
ax[0].set_ylabel('u(x) [m]')
ax[0].grid(True, alpha=0.3)

ax[1].plot(t, y, label='y(t)=u(x(t))', linewidth=1.5)
ax[1].plot(t, x1, label='x1(t): masa no suspendida', linewidth=1.2)
ax[1].plot(t, x2, label='x2(t): masa suspendida', linewidth=1.2)
ax[1].set_title('Respuesta temporal del sistema 2 GDL')
ax[1].set_xlabel('Tiempo [s]')
ax[1].set_ylabel('Desplazamiento [m]')
ax[1].grid(True, alpha=0.3)
ax[1].legend()

plt.tight_layout()
plt.show()

## 10. Relacion con los objetivos del README

1. Se plantea el modelo dinamico continuo de 2 GDL.
2. Se implementan las ecuaciones diferenciales acopladas.
3. Se simula la respuesta temporal para entrada senoidal de base.
4. Se habilita el analisis de sensibilidad variando m1, m2, k1, k2 y c.
5. Se observan amplitudes y respuesta de x2(t) como indicador de confort.